# AutoSentinel AI — Automotive Recall Intelligence Pipeline

Welcome to the official machine learning and safety intelligence notebook for the **AutoSentinel AI** platform. This notebook details the end-to-end research, engineering, and model training workflow for explaining and predicting automotive safety hazards from NHTSA recall campaigns.

## 🏗️ Pipeline Architecture

1. **Exploratory Data Analysis (EDA)**: Profiling recall volumes, component distributions, and average risk indices.
2. **Preprocessing & Feature Engineering**: Converting unstructured NLP text and categorical manufacturer variables.
3. **XGBoost Classifier**: Multi-class extreme gradient boosted decision tree classifier maps variables to safety risk labels (`Low`, `Medium`, `High`, `Critical`).
4. **Explainable AI (SHAP)**: Marginal local Shapley values calculated over TF-IDF sparse spaces to explain every prediction verdict.
5. **Semantic Dense Embeddings**: SentenceTransformer (`all-MiniLM-L6-v2`) encodes descriptions into 384-dimensional dense vectors to support high-dimensional cosine-similarity search.

## 🛠️ Step 1: System Imports and Library Load

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
from pathlib import Path

# Machine Learning and Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import xgboost as xgb

# Explainability and Semantic Search
import shap
from sentence_transformers import SentenceTransformer

# Setup plot styling
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.family"] = "sans-serif"

## 📊 Step 2: Exploratory Data Analysis (EDA)

Let's load the engineered NHTSA vehicle safety recalls dataset and inspect its structure, dimensions, and variables.

In [ ]:
# Load processed dataset
data_path = Path("../data/processed/ml_ready_vehicle_recalls.csv")
if not data_path.exists():
    # Fallback to local execution path inside workspace
    data_path = Path("data/processed/ml_ready_vehicle_recalls.csv")

df = pd.read_csv(data_path, low_memory=False)
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(3)

In [ ]:
# 1. Plot Risk Label Distributions
plt.figure(figsize=(8, 5))
sns.countplot(x="risk_label", data=df, order=["Low", "Medium", "High", "Critical"], palette="viridis")
plt.title("NHTSA Campaign Safety Risk Tier Distributions", fontsize=14, fontweight="bold")
plt.xlabel("Safety Risk Tier", fontweight="bold")
plt.ylabel("Campaign Count", fontweight="bold")
plt.show()

In [ ]:
# 2. Plot Top Component Defect Categories
plt.figure(figsize=(10, 6))
comp_order = df["component"].value_counts().index[:10]
sns.countplot(y="component", data=df, order=comp_order, palette="mako")
plt.title("Top 10 Monitored Automotive Defect Systems", fontsize=14, fontweight="bold")
plt.xlabel("Recall Campaign Volume", fontweight="bold")
plt.ylabel("Vehicle System Category", fontweight="bold")
plt.show()

In [ ]:
# 3. Plot Safety Index rankings for manufacturers
plt.figure(figsize=(12, 6))
mfr_sri = df.groupby("manufacturer")["defect_severity_score"].mean().sort_values(ascending=False).head(10)
sns.barplot(x=mfr_sri.values, y=mfr_sri.index, palette="rocket")
plt.title("Highest Safety Hazard Indexes by Manufacturer Make", fontsize=14, fontweight="bold")
plt.xlabel("Weighted Safety Index Quotient", fontweight="bold")
plt.ylabel("Manufacturer Make", fontweight="bold")
plt.show()

## ⚙️ Step 3: Feature Engineering & Column Transformation

We need to engineer text vectorizations (using a sparse TF-IDF space) for defect descriptions and categorical column transforms (using one-hot encodes) for manufacturers, components, and vehicle years.

In [ ]:
# Prepare features and targets
text_feature = "summary"
categorical_features = ["manufacturer", "component", "vehicle_age"]
target_column = "risk_label"

# Ensure clean strings
df["summary"] = df["summary"].fillna("").astype(str)
df["manufacturer"] = df["manufacturer"].fillna("Unknown").astype(str)
df["component"] = df["component"].fillna("Other").astype(str)
df["vehicle_age"] = df["vehicle_age"].fillna(5).astype(int)

# Split into Train and Test
X = df[[text_feature] + categorical_features]
y = df[target_column]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Target classes encoded: {dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))}")

In [ ]:
# Create structured preprocessor pipeline
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('cat', categorical_transformer, categorical_features)
], remainder='drop')

# Extract text embeddings via TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=1000, stop_words='english', ngram_range=(1, 2))

# Fit pipelines on training datasets
X_train_cat = preprocessor.fit_transform(X_train)
X_train_text = tfidf_vectorizer.fit_transform(X_train[text_feature]).toarray()

# Combine sparse features and categorical inputs
X_train_final = np.hstack((X_train_cat, X_train_text))

X_test_cat = preprocessor.transform(X_test)
X_test_text = tfidf_vectorizer.transform(X_test[text_feature]).toarray()
X_test_final = np.hstack((X_test_cat, X_test_text))

print(f"Final Feature Matrix dimensions: {X_train_final.shape}")

## 🌲 Step 4: XGBoost Model Training and Validation

In [ ]:
# Initialize multiclass Extreme Gradient Boosting model
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=len(label_encoder.classes_),
    n_estimators=150,
    max_depth=6,
    learning_rate=0.15,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

print("Training XGBoost multiclass safety classifier...")
xgb_model.fit(X_train_final, y_train)
print("Model fit complete.")

In [ ]:
# Predict on test dataset
y_pred = xgb_model.predict(X_test_final)
print(f"Out-of-sample Prediction Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%")

In [ ]:
# Print out precise classification report matrix
report = classification_report(y_test, y_pred, target_names=label_encoder.classes_)
print(report)

In [ ]:
# Plot Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_)
plt.title("AutoSentinel AI Confusion Matrix", fontsize=13, fontweight="bold")
plt.xlabel("Predicted Safety Risk Tier")
plt.ylabel("Actual Safety Risk Tier")
plt.show()

## 🔍 Step 5: Explainable AI with SHAP (Shapley Additive exPlanations)

SHAP maps local marginal Shapley values for individual token inputs, allowing us to explain the model's prediction directly on natural language defect text.

In [ ]:
# Initialize TreeExplainer for XGBoost
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_final[:5])

print("SHAP Local Explainer armed.")
print(f"SHAP Explanations array shape: {np.shape(shap_values)}")

## ⚡ Step 6: Dense Semantic Vector Embeddings for Cosine similarity

We use **SentenceTransformers** (`all-MiniLM-L6-v2`) to encode defect descriptions into high-dimensional vector spaces and compute cosine similarity matches.

In [ ]:
print("Initializing sentence embedder core...")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sample_defects = [
    "The steering wheel lockups during driving at high speeds.",
    "The high voltage battery pack catches fire while charging."
]

embeddings = embedder.encode(sample_defects)
print(f"Dense Vector space size: {embeddings.shape[0]} arrays of dimensions {embeddings.shape[1]}")

In [ ]:
# Compute sample cosine similarities
query = "steering wheel locking up unexpectedly"
query_embedding = embedder.encode([query])[0]

for index, text in enumerate(sample_defects):
    sim = np.dot(query_embedding, embeddings[index]) / (np.linalg.norm(query_embedding) * np.linalg.norm(embeddings[index]))
    print(f"Cosine Similarity Match: {sim*100:.2f}% | Text: {text}")

## 💾 Step 7: Serialize Model Artifacts

Finally, we serialize our pipelines, labels, models, and embeddings directly to the `/artifacts` target folder so the live Uvicorn API server can serve them.

In [ ]:
artifacts_dir = Path("../artifacts")
if not artifacts_dir.exists():
    artifacts_dir = Path("artifacts")

artifacts_dir.mkdir(exist_ok=True)

# 1. Save XGBoost Classifier
with open(artifacts_dir / "xgboost_model.pkl", "wb") as f:
    pickle.dump(xgb_model, f)

# 2. Save Label Encoder
with open(artifacts_dir / "label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

# 3. Save TF-IDF Vectorizer
with open(artifacts_dir / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)

# 4. Save Structured Preprocessor
with open(artifacts_dir / "structured_preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)

print("All pipeline objects serialized successfully.")